# 02 — MetaOrchestrator

**Decompose a goal, spawn the specialists to do it — at runtime.**

The autonomy layer turns a one-line goal into a running multi-agent system. A *planner* agent breaks the goal into subtasks; the `MetaOrchestrator` finds or spawns an agent for each one, runs them in sequence, and aggregates the results. Successful agent specs are persisted to memory so a later run can rehydrate them instead of re-spawning.

This notebook composes: `BaseAgent` (the planner) + `ToolRegistry` + `AgentFactory` + `MetaOrchestrator` + `LocalMemoryStore`.

**Prerequisites:** a `.env` with `LLM_API_KEY` and `LLM_MODEL`.

## 1. Load config

In [1]:
from orqest import load_config

config = load_config()
print(f"Model: {config.llm_model}")

Model: openai:gpt-5.2


## 2. The planner

The planner is an ordinary `BaseAgent` whose output type is `TaskDecomposition` — a goal, an ordered list of `SubTask`s, and the reasoning behind the split. Nothing special: it's a typed agent like any other.

In [2]:
from orqest.agents import BaseAgent, GlobalState
from orqest.autonomy.meta import TaskDecomposition


class PlannerAgent(BaseAgent[GlobalState, TaskDecomposition]):
    def __init__(self, model, api_key=None):
        super().__init__(
            agent_name="planner",
            system_prompt=(
                "You decompose a high-level goal into 2-4 concrete subtasks. "
                "Each subtask needs a short snake_case name, a one-sentence "
                "description, and requires_agent=True. Keep the list minimal."
            ),
            output_type=TaskDecomposition,
            model=model,
            api_key=api_key,
        )

    async def _run_implementation(self, state: GlobalState, **kwargs) -> TaskDecomposition:
        result = await self.call_model(state.get_latest_message("user"), state)
        return result.output


planner = PlannerAgent(model=config.llm_model, api_key=config.llm_api_key)

## 3. Registry, factory, orchestrator

- `ToolRegistry` — the shared tool namespace spawned agents resolve against. Empty here; the spawned agents in this demo are pure reasoners.
- `AgentFactory` — hydrates an `AgentSpec` into a live `DynamicAgent` (it builds the output model from JSON Schema via `pydantic.create_model`).
- `MetaOrchestrator` — ties the planner, factory, and registry together. `max_subtasks` bounds the work.

In [3]:
from orqest.autonomy import ToolRegistry, AgentFactory, MetaOrchestrator

registry = ToolRegistry()
factory = AgentFactory(
    registry,
    default_model=config.llm_model,
    api_key=config.llm_api_key,
)
meta = MetaOrchestrator(planner, factory, registry, max_subtasks=4)

## 4. Solve a goal

`solve()` runs the whole loop: decompose → for each subtask find-or-spawn an agent → execute → aggregate. It returns an `ExecutionResult`.

In [4]:
result = await meta.solve(
    "Design a command-line tool that converts CSV files to JSON: name its "
    "subcommands, list the global flags, and write one usage example."
)

print(f"goal succeeded: {result.success}")
print(f"duration: {result.total_duration_ms:.0f} ms")
print(result.summary)

goal succeeded: True
duration: 13804 ms
Goal: Design a command-line tool that converts CSV files to JSON: name its subcommands, list the global flags, and write one usage example.
  [OK] define_subcommands (via define_subcommands)
  [OK] specify_global_flags (via specify_global_flags)
  [OK] provide_usage_example (via provide_usage_example)


In [5]:
for r in result.subtask_results:
    status = "ok" if r.success else "FAILED"
    print(f"[{status}] {r.subtask_name}  (agent={r.agent_used}, spawned={r.was_spawned})")
    if r.success:
        print(f"        {str(r.output)[:140]}")

[ok] define_subcommands  (agent=define_subcommands, spawned=True)
        result='1. convert: Converts a CSV file to a JSON file. Supports specifying input and output files, and configurable options (such as delimi
[ok] specify_global_flags  (agent=specify_global_flags, spawned=True)
        result='Global flags (options) available across subcommands—particularly relating to I/O, formatting, and behavior controls—typically includ
[ok] provide_usage_example  (agent=provide_usage_example, spawned=True)
        result='csvtool convert --input data.csv --output data.json --delimiter , --encoding utf-8' confidence=1.0


## 5. What got spawned

Every agent the orchestrator created at runtime is kept in `spawned_agents`, keyed by subtask name — so a later subtask in the same run can reuse one without re-spawning.

In [6]:
for name, ag in meta.spawned_agents.items():
    print(f"{name:30s} -> {type(ag).__name__}")

define_subcommands             -> DynamicAgent
specify_global_flags           -> DynamicAgent
provide_usage_example          -> DynamicAgent


## 6. Memory-backed reuse across runs

Pass a `MemoryStore` and the orchestrator dual-writes every spawned spec: an episodic mirror and a procedural `Skill` (keyed on the subtask name). A *fresh* orchestrator sharing the same store rehydrates a matching subtask from memory instead of spawning it again.

A real LLM planner's subtask names vary run to run, which would hide the recall path — so here we pin the decomposition with a fixed planner to make reuse observable. The tell: the store stops growing on run 2, because the recall path returns *before* the spawn-and-persist step.

In [7]:
import tempfile, os
from orqest.memory import LocalMemoryStore
from orqest.autonomy.meta import SubTask

store = LocalMemoryStore(os.path.join(tempfile.mkdtemp(), "meta_memory.db"))


class FixedPlanner(BaseAgent[GlobalState, TaskDecomposition]):
    """Returns a pinned decomposition — no LLM call — so both runs match."""
    def __init__(self):
        super().__init__(
            agent_name="fixed_planner", system_prompt="plan",
            output_type=TaskDecomposition,
            model=config.llm_model, api_key=config.llm_api_key,
        )

    async def _run_implementation(self, state, **kwargs) -> TaskDecomposition:
        return TaskDecomposition(
            goal="ship a small CLI tool",
            subtasks=[
                SubTask(name="design_cli",
                        description="Design the CLI surface and arguments.",
                        requires_agent=True),
                SubTask(name="write_readme",
                        description="Draft a concise README for the tool.",
                        requires_agent=True),
            ],
            reasoning="Design, then document.",
        )


# Run 1 — a fresh orchestrator spawns both agents and persists their specs.
meta_a = MetaOrchestrator(FixedPlanner(), factory, registry, memory=store)
await meta_a.solve("ship a small CLI tool")
count_after_run_1 = await store.count()
print(f"after run 1: store holds {count_after_run_1} entries")

# Run 2 — a *different* orchestrator, same store. Its in-process cache is
# empty, so a matching subtask must be rehydrated from memory.
meta_b = MetaOrchestrator(FixedPlanner(), factory, registry, memory=store)
await meta_b.solve("ship a small CLI tool")
count_after_run_2 = await store.count()
print(f"after run 2: store holds {count_after_run_2} entries", end="  ")
print("(stable -> run 2 reused the specs from memory)"
      if count_after_run_2 == count_after_run_1 else "(grew -> no reuse)")

after run 1: store holds 4 entries


after run 2: store holds 4 entries  (stable -> run 2 reused the specs from memory)


In [8]:
# The persisted procedural skills — each keyed on its subtask name (trigger).
skills = await store.list_recent(memory_type="procedural")
for s in skills:
    spec = s.structured_content.get("spec", {}) if s.structured_content else {}
    print(f"skill trigger={s.content!r}  ->  spec.name={spec.get('name')!r}")

skill trigger='write_readme'  ->  spec.name='write_readme'
skill trigger='design_cli'  ->  spec.name='design_cli'


## What's happening under the hood

1. `solve(goal)` runs the planner once to get a `TaskDecomposition`, then walks the subtasks in order (bounded by `max_subtasks`).
2. `_find_or_spawn` checks, in order: the in-process `spawned_agents` cache → procedural memory → episodic memory → spawn fresh. All three lookup paths key on `subtask.name`.
3. A freshly spawned agent's spec is dual-written to memory — an episodic mirror plus a procedural `Skill` whose `trigger` is the subtask name.
4. Each subtask runs through `_execute_subtask`, which fires the `HookRunner` lifecycle (`before` / `after` / `on_error`) — so a `WatchdogHook` or `EventBusPublishHook` plugs straight in.
5. With a `MetacognitionConfig`, a low-confidence subtask result triggers re-decomposition of the *remaining* subtasks — the orchestrator re-plans mid-run. (See notebook 01 for the confidence machinery.)

The factory builds each agent's output model from a JSON Schema at runtime via `pydantic.create_model` — which is why an `AgentSpec` is fully serializable and an LLM can emit one as structured output.